<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/02f_hyperparameter_tuning_optuna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Neural Network Hyperparameters

This notebook covers the basic ideas approach for automating the fine-tuning of neural network hyperparameters using [Optuna](https://optuna.org/).

In [ ]:
%%bash

pip install --upgrade optuna torchmetrics

In [ ]:
import numpy as np
from sklearn import compose, datasets, model_selection, pipeline, preprocessing

import optuna
import torch
from torch import nn, optim, utils
import torchmetrics

## Verifying availability of GPU(s)

In [ ]:
# check that torch version has support for cuda
print(torch.__version__)

In [ ]:
%%bash

# check that GPUs are physically available
nvidia-smi

In [ ]:
# check that PyTorch can find the GPUs
print(torch.cuda.is_available())

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(DEVICE)

## Loading the data

In [ ]:
covtype_dataset = datasets.fetch_covtype(
    as_frame=True
)

In [ ]:
print(covtype_dataset["DESCR"])

In [ ]:
covtype_features_df = covtype_dataset["data"]
covtype_target_df = (
    covtype_dataset.get("target")
                   .to_frame()
)

## Preparing the data

### Train/val/test split

In [ ]:
RANDOM_STATE = np.random.RandomState(42)


train_features_df, _val_test_features_df, train_target_df, _val_test_target_df = (
    model_selection.train_test_split(
        covtype_features_df,
        covtype_target_df,
        test_size=0.30,
        shuffle=True,
        stratify=covtype_target_df,
        random_state=RANDOM_STATE
    )
)

val_features_df, test_features_df, val_target_df, test_target_df = (
    model_selection.train_test_split(
        _val_test_features_df,
        _val_test_target_df,
        test_size=2/3,
        shuffle=True,
        stratify=_val_test_target_df,
        random_state=RANDOM_STATE
    )
)


### Features and target preparation

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
    return torch.tensor(arr, dtype=dtype)


prepare_covtype_features = pipeline.make_pipeline(
    compose.make_column_transformer(
        (
            "passthrough",
            compose.make_column_selector(
                pattern="^Wilderness_Area_|^Soil_Type_"
            )
        ),
        force_int_remainder_cols=False,
        n_jobs=-1,
        remainder=preprocessing.QuantileTransformer(
            output_distribution="normal",
            random_state=RANDOM_STATE,
        )
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
    )
)

prepare_covtype_target = pipeline.make_pipeline(
    preprocessing.OrdinalEncoder(
        categories=[
            [1, 2, 3, 4, 5, 6, 7]
        ],
    ),
    preprocessing.FunctionTransformer(
        func=array_to_tensor,
        kw_args={
            "dtype": torch.int64
        }
    ),
    preprocessing.FunctionTransformer(
        func=torch.squeeze,
    )
)



In [ ]:
X_train = prepare_covtype_features.fit_transform(train_features_df)
X_val = prepare_covtype_features.transform(val_features_df)
X_test = prepare_covtype_features.transform(test_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
y_train = prepare_covtype_target.fit_transform(train_target_df)
y_val = prepare_covtype_target.transform(val_target_df)
y_test = prepare_covtype_target.transform(test_target_df)

In [ ]:
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

### Datasets

In [ ]:
train_dataset = utils.data.TensorDataset(X_train, y_train)
val_dataset = utils.data.TensorDataset(X_val, y_val)
test_dataset = utils.data.TensorDataset(X_test, y_test)

### DataLoaders

In [ ]:
TRAIN_DATA_LOADER = (
    utils.data
         .DataLoader(
             train_dataset,
             num_workers=2,
             batch_size=128,
             shuffle=True,
             persistent_workers=True,
             pin_memory=True,
             prefetch_factor=2,
             drop_last=True,
         )
)

VAL_DATA_LOADER = (
    utils.data
         .DataLoader(
             val_dataset,
             num_workers=2,
             batch_size=128,
             shuffle=False,
             persistent_workers=True,
             pin_memory=True,
             prefetch_factor=2,
             drop_last=True,
         )
)

TEST_DATA_LOADER = (
    utils.data
         .DataLoader(
             test_dataset,
             num_workers=2,
             batch_size=128,
             shuffle=False,
             persistent_workers=True,
             pin_memory=True,
             prefetch_factor=2,
             drop_last=True,
         )
)

## Training and evaluation functions


In [ ]:
def train(
    model_fn,
    criterion,
    optimizer,
    train_data_loader,
    n_epochs,
    ):

    model_fn.train()
    for epoch in range(n_epochs):
        for i, (X_batch, y_batch) in enumerate(train_data_loader):

            # move batches to device
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            # forward pass
            y_pred = model_fn(X_batch)
            train_loss = criterion(y_pred, y_batch)

            # backward pass
            train_loss.backward()

            # gradient descent step
            optimizer.step()
            optimizer.zero_grad()

In [ ]:
def evaluate_model_fn(model_fn, data_loader, metric):
    model_fn.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            y_pred = model_fn(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [ ]:
LOSS_FN = nn.CrossEntropyLoss()

N_CLASSES = y_train.unique().size(0)
EVALUATION_METRIC = torchmetrics.Accuracy(
    num_classes=N_CLASSES,
    task="multiclass",
).to(DEVICE)

## Defining our hyperparameter tuning objective function

When tuning hyperparameters of our neural network we need to have a function that

1. takes some representation of hyperparameter values as an input,
2. internally trains a neural network with those hyperparameter values,
3. evaluates the trained model using some metric,
4. returns the value of the evaluation performance metric.

With such a function the process of tuning hyperparameters involves finding values of the hyperparameters that optimize this objective function.

Such an objective function that would work with Optuna would have the following pseudo-code.

```python
def objective_fn(trial: optuna.Trial) -> torch.float32:
    
    # suggest hyperparameters
    suggested_model_hyperparameters = {...}
    suggested_optimizer_hyperparameters = {...}

    # build a model with relevant hyperparameters
    model_fn = make_model_fn(suggested_model_hyperparameters)
    model_fn = model_fn.to(DEVICE)

    # build an optimizer with relevant hyperparameters
    optimizer = make_optimizer(model_fn.parameters(suggested_optimizer_hyperparameters),

    # train the model using the optimizer and loss
    _ train(model_fn, optimizer, LOSS_FN, TRAIN_DATA_LOADER, N_EPOCHS)

    # evaluate the trained model
    computed_val_metric = evaluate(model_fn, VAL_DATA_LOADER, EVALUATION_METRIC)

    return computed_val_metric
```


### Defining a `make_model_fn`

In [ ]:
INPUT_SIZE = X_train.size(1)

In [ ]:
class MLPClassifier(nn.Module):

    def __init__(self, input_size, hidden_layer_sizes, n_classes):
        super().__init__()

        # create the hidden layers
        modules = nn.ModuleList([nn.Flatten()])
        for hidden_layer_size in hidden_layer_sizes:
            modules.append(nn.Linear(input_size, hidden_layer_size))
            modules.append(nn.ReLU())
            input_size = hidden_layer_size

        # define the output layer for the classifier
        modules.append(nn.Linear(input_size, n_classes))

        # create the MLP from the modules
        self.mlp = nn.Sequential(*modules)

    def forward(self, X):
        return self.mlp(X)


In [ ]:
def make_model_fn(model_fn_hyperparameters):
    return MLPClassifier(
        input_size=INPUT_SIZE,
        n_classes=N_CLASSES,
        **model_fn_hyperparameters,
    )


### Defining a `make_optimizer_fn`

In [ ]:
def make_optimizer(parameters, optimizer_hyperparameters):
    return optim.SGD(
        parameters,
        **optimizer_hyperparameters,
    )

### Defining a `make_objective_fn`

Rather than using global variables to control the internal behavior of the objective function, a better approach is to define a `make_objective_fn` that takes training and validation data loaders, the loss function, the evaluation metric, etc as inputs and returns the objective function.

In [ ]:
def make_objective_fn(
    train_data_loader,
    val_data_loader,
    criterion,
    evaluation_metric,
    n_epochs,
    ):

    def objective_fn(trial):

        # suggest hyperparameters
        n_hidden_layers = trial.suggest_int(
            name="n_hidden_layers",
            low=1,
            high=10,
            step=1,
        )

        n_hidden_units = trial.suggest_int(
            name="n_hidden_units",
            low=10,
            high=100,
            step=10,
        )

        lr = trial.suggest_float(
            name="lr",
            low=1e-5,
            high=1e-1,
            log=True,
        )

        # build a model with relevant hyperparameters
        model_fn_hyperparameters = {
            "hidden_layer_sizes": [n_hidden_units] * n_hidden_layers,
        }
        model_fn = make_model_fn(model_fn_hyperparameters)
        model_fn = model_fn.to(DEVICE)

        # build and optimizer with relevant hyperparameters
        optimizer = make_optimizer(
            model_fn.parameters(),
            lr
          )

        # train the model using the optimizer and loss
        _ = train(
            model_fn,
            criterion,
            optimizer,
            train_data_loader,
            n_epochs=n_epochs,
        )

        # evaluate the trained model
        computed_val_metric = evaluate_model_fn(
            model_fn,
            val_data_loader,
            evaluation_metric,
        )

        return computed_val_metric.item()

    return objective_fn

In [ ]:
objective_fn = make_objective_fn(
    train_data_loader=TRAIN_DATA_LOADER,
    val_data_loader=VAL_DATA_LOADER,
    criterion=LOSS_FN,
    evaluation_metric=EVALUATION_METRIC,
    n_epochs=10,
)

## Sampling hyperparameters (intelligently!)

Now that we have an objective function that takes an `optuna.Trial` and returns our evaluation metric we need a way to create some number of trials that we can evaluate with the hope that at least one of the trials will give us better performance.

Optuna uses the [Tree-structured Parzen Estimator (TPE)](https://arxiv.org/abs/2304.11127) algorithm for hyperparameter optimization. TPE is a sequential model-based optimization method that learns from previous trials to guide future searches toward more promising hyperparameter values. While the process begins with random samples, the algorithm continuously updates its internal model based on past performance, allowing it to concentrate exploration on the most promising regions of the search space. As a result, the algorithm typically discovers better hyperparameter configurations than simple random search within the same computational budget.

In [ ]:
_ = torch.manual_seed(42)
SAMPLER = optuna.samplers.TPESampler(seed=42)

basic_study = optuna.create_study(
    direction="maximize",  # higher accuracy is better!
    sampler=SAMPLER,
)

In [ ]:
basic_study.optimize(
    objective_fn,
    n_trials=5
)

In [ ]:
basic_study.best_params

In [ ]:
basic_study.best_value

## Pruning poorly performing hyper-parameter combinations

When tuning hyperparameters, it’s often clear early in training that a given trial performs poorly-for example, if the loss spikes or validation accuracy stagnates after the first few epochs. To save time and compute, you can stop these unpromising trials early. This can be done by raising the `optuna.TrialPruned` exception during training, which tells Optuna to discard the trial entirely. Pruning poor trials in this way makes the search more efficient and prevents noisy or failed runs from biasing Optuna’s optimization strategy.

### Modifying our training function

In [ ]:
def train(
    trial,
    model_fn,
    criterion,
    optimizer,
    train_data_loader,
    val_data_loader,
    evaluation_metric,
    n_epochs,
    ):

    for epoch in range(n_epochs):
        model_fn.train()
        for i, (X_batch, y_batch) in enumerate(train_data_loader):

            # move batches to device
            X_batch = X_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            # forward pass
            y_pred = model_fn(X_batch)
            train_loss = criterion(y_pred, y_batch)

            # backward pass
            train_loss.backward()

            # gradient descent step
            optimizer.step()
            optimizer.zero_grad()

        # evaluate the performance of the trained model
        average_val_metric = evaluate_model_fn(
            model_fn,
            val_data_loader,
            evaluation_metric,
        )

        trial.report(average_val_metric.item(), epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return average_val_metric



### Modifying our `make_objective_fn`

In [ ]:
def make_objective_fn(
    train_data_loader,
    val_data_loader,
    criterion,
    evaluation_metric,
    n_epochs,
    ):

    def objective_fn(trial):

        # suggest hyperparameters
        n_hidden_layers = trial.suggest_int(
            name="n_hidden_layers",
            low=1,
            high=10,
            step=1,
        )

        n_hidden_units = trial.suggest_int(
            name="n_hidden_units",
            low=10,
            high=100,
            step=10,
        )

        lr = trial.suggest_float(
            name="lr",
            low=1e-5,
            high=1e-1,
            log=True,
        )

        # build a model with relevant hyperparameters
        model_fn_hyperparameters = {
            "hidden_layer_sizes": [n_hidden_units] * n_hidden_layers,
        }
        model_fn = make_model_fn(model_fn_hyperparameters)
        model_fn = model_fn.to(DEVICE)

        # build and optimizer with relevant hyperparameters
        optimizer = make_optimizer(
            model_fn.parameters(),
            lr
          )

        # train and evaluate the model with access to the trial object
        computed_val_metric = train(
            trial,
            model_fn,
            criterion,
            optimizer,
            train_data_loader,
            val_data_loader,
            evaluation_metric,
            n_epochs,
        )

        return computed_val_metric.item()

    return objective_fn

### Defining our pruning strategy

In [ ]:
PRUNER = optuna.pruners.MedianPruner(
    n_startup_trials=5,
    n_warmup_steps=10,
    interval_steps=1
)

study_with_pruning = optuna.create_study(
    direction="maximize",
    sampler=SAMPLER,
    pruner=PRUNER
)

In [ ]:
objective_fn = make_objective_fn(
    train_data_loader=TRAIN_DATA_LOADER,
    val_data_loader=VAL_DATA_LOADER,
    criterion=LOSS_FN,
    evaluation_metric=EVALUATION_METRIC,
    n_epochs=10,
)

In [ ]:
study_with_pruning.optimize(objective_fn, n_trials=10)

### Exercise:

After optimizing your hyperparameters your would typically want to re-train your model on the combined training and validation datasets using the best hyperparameters found, and then evaluate the resulting model using the testing dataset.

Merge the training and validation datasets and then re-train your model using the best hyperparameters and evaluate the trained model using the testing dataset.

In [ ]:
# INSERT YOUR CODE HERE!

### Exercise:

Train a [`HistGradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html) from the `ensemble` package of Scikit-Learn using default parameters. Evaluate the performance of your model using the testing dataset. Does this model outperform your tuned neural network?

In [ ]:
# INSERT YOUR CODE HERE!